# Opik dataset: CSV ↔ Opik

Uses `opik-ai-evaluations/.venv` (works even if Jupyter picks system Python).

One-time terminal setup:

```bash
cd opik-ai-evaluations
python3 -m venv .venv
.venv/bin/pip install -U pip opik pandas ipykernel python-dotenv
.venv/bin/python -m ipykernel install --user --name=opik-eval --display-name="Python (opik-eval)"
```

1. Run **bootstrap**.
2. Adjust **config** (`DATASET_NAME`, `CSV_PATH`).
3. Run **upload** to push the configured CSV to Opik.
4. Optionally run **export** to download an existing Opik dataset to CSV.


In [34]:
import sys
from pathlib import Path

_repo = Path.cwd()
if not (_repo / "notebook_env.py").exists():
    _fallback = Path("/home/nxtwave/Repositories/opik-ai-evaluations")
    _repo = _fallback if (_fallback / "notebook_env.py").exists() else _repo
sys.path.insert(0, str(_repo))

from notebook_env import activate

REPO_ROOT = activate()

import opik
import pandas as pd

print("OK —", sys.executable)
print("repo:", REPO_ROOT)
print("opik", opik.__version__, "| pandas", pd.__version__)


Note: Jupyter is not using the .venv interpreter (prefix=/usr).
Loaded packages from /home/nxtwave/Repositories/opik-ai-evaluations/.venv/lib/python3.12/site-packages
Loaded environment from /home/nxtwave/Repositories/opik-ai-evaluations/.env
OK — /usr/bin/python
repo: /home/nxtwave/Repositories/opik-ai-evaluations
opik 2.2.100 | pandas 3.0.6


## Config

Default: upload **DSA coding tutor** export (`ai_tutor_coding_practise_dsa_prompts_20261015.csv`). Switch config for IDE grader (`ai_eval_ide_prod_v2.csv`).


In [35]:
import importlib
import os

import notebook_env
from opik import Opik

importlib.reload(notebook_env)
notebook_env.load_env_file(override=True)

os.environ.setdefault("OPIK_URL_OVERRIDE", "https://3.110.54.210/api")
os.environ.setdefault("OPIK_CHECK_TLS_CERTIFICATE", "false")
os.environ.setdefault("OPIK_PROJECT_NAME", "AI Evaluations")

# --- Pick export to upload ---
DATASET_NAME = "ai_tutor_coding_practise_dsa_prompts"
CSV_PATH = REPO_ROOT / "data" / "exports" / "ai_tutor_coding_practise_dsa_prompts_20261015.csv"

# IDE grader example:
# DATASET_NAME = "ai_eval_ide_prod_v2"
# CSV_PATH = REPO_ROOT / "data" / "exports" / "ai_eval_ide_prod_v2.csv"

MAX_ROWS = None  # e.g. 100 to cap rows; None = upload entire CSV
UPLOAD_CHUNK_SIZE = 25

# Replace all items in Opik before upload (use when fixing wrong row count)
CLEAR_EXISTING_DATASET = False

# If True, upload even when the dataset already has items (may duplicate rows)
FORCE_UPLOAD = False

opik_client = Opik(project_name=os.environ["OPIK_PROJECT_NAME"])
print("project:", os.environ["OPIK_PROJECT_NAME"])
print("dataset:", DATASET_NAME)
print("csv:", CSV_PATH)
print("csv exists:", CSV_PATH.is_file())


project: AI Evaluations
dataset: ai_tutor_coding_practise_dsa_prompts
csv: /home/nxtwave/Repositories/opik-ai-evaluations/data/exports/ai_tutor_coding_practise_dsa_prompts_20261015.csv
csv exists: True


## Upload CSV → Opik

Uses `get_or_create_dataset` (not `get_dataset`) so a missing dataset is created automatically.

Skips upload when `dataset_items_count > 0` unless `FORCE_UPLOAD` is `True`.

Opik requires each item's `id` to be **UUID v7**. **Tutor** exports: use CSV `id` as-is. **IDE grader** exports: rename CSV `id` → `user_response_id` and derive Opik `id` from `submission_id` + `submission_created_at`.


In [36]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

from datetime import datetime, timezone

from opik.id_helpers import is_valid_uuid_v7, uuid4_to_uuid7

header_cols = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
string_cols = [
    "id", "submission_id", "user_id", "question_id",
    "conversation_id", "bot_id", "code_id",
]

dtype = {col: str for col in string_cols if col in header_cols}

df = pd.read_csv(CSV_PATH, dtype=dtype, keep_default_na=True).fillna("")

def _submission_to_opik_id(submission_id: str, created_at: str) -> str:
    if not submission_id:
        raise ValueError("missing submission_id")
    try:
        ts = datetime.fromisoformat(created_at.replace("Z", "+00:00"))
    except (ValueError, AttributeError):
        ts = datetime.now(timezone.utc)
    return str(uuid4_to_uuid7(ts, submission_id))

if "submission_id" in df.columns and "submission_created_at" in df.columns:
    upload_df = df.rename(columns={"id": "user_response_id"}).copy()
    upload_df["id"] = upload_df.apply(
        lambda row: _submission_to_opik_id(row["submission_id"], row["submission_created_at"]),
        axis=1,
    )
    upload_mode = "ide_grader"
elif "id" in df.columns:
    upload_df = df.copy()
    bad_ids = [
        idx for idx, value in enumerate(upload_df["id"].tolist())
        if not is_valid_uuid_v7(str(value))
    ]
    if bad_ids:
        raise ValueError(f"{len(bad_ids)} rows have ids that are not UUID v7 (first indexes: {bad_ids[:5]})")
    upload_mode = "tutor_trace"
else:
    raise ValueError("Unrecognized CSV shape: need tutor `id` or IDE `submission_id` + `submission_created_at`")

if MAX_ROWS is not None:
    upload_df = upload_df.head(int(MAX_ROWS)).copy()

print("upload_mode:", upload_mode)
print("loaded rows:", len(upload_df))
print("columns:", upload_df.columns.tolist())
upload_df.head(2)


upload_mode: tutor_trace
loaded rows: 100
columns: ['ref_output', 'input', 'conversation_id', 'current_prompt', 'question_content', 'bot_id', 'code_id', 'id']


,ref_output,input,conversation_id,current_prompt,question_content,bot_id,code_id,id
0,"To fix the issue, remove the period after `{se...","[{""role"": ""user"", ""content"": ""Hi my name is T ...",9fcd006a0e2e46f991ebacd4f4c2cb1a,# Role\n\nYou are a senior **Data Structures a...,## Coding Question: you are given an incomplet...,1e12ca42-92a0-4ca9-a192-f374d283fa1f,,019a168c-b356-78d9-8bef-bffdef5a74a7
1,"To fix the issue, remove the period after `{se...","[{""role"": ""user"", ""content"": ""Hi my name is T ...",9fcd006a0e2e46f991ebacd4f4c2cb1a,# Role\n\nYou are a senior **Data Structures a...,## Coding Question: you are given an incomplet...,1e12ca42-92a0-4ca9-a192-f374d283fa1f,1e558a7a-a261-4e21-af4e-80c6657290c7,019a168c-b355-71f5-8b09-239b49839e9a


In [37]:
dataset = opik_client.get_or_create_dataset(DATASET_NAME)

print("name:", dataset.name)
print("id:", dataset.id)
current_size = dataset.dataset_items_count or 0
print("item count (before upload):", current_size)

if CLEAR_EXISTING_DATASET and current_size > 0:
    print(f"Clearing {current_size} existing items from '{DATASET_NAME}'...")
    dataset.clear()
    current_size = 0

should_upload = FORCE_UPLOAD or current_size == 0

if not should_upload:
    print(
        f"⏭️ Skipping upload: '{DATASET_NAME}' already has {current_size} items. "
        "Set FORCE_UPLOAD=True to upload anyway."
    )
else:
    total = len(upload_df)
    uploaded = 0
    for start in range(0, total, UPLOAD_CHUNK_SIZE):
        chunk = upload_df.iloc[start : start + UPLOAD_CHUNK_SIZE]
        dataset.insert_from_pandas(dataframe=chunk)
        uploaded += len(chunk)
        print(f"uploaded {uploaded}/{total} rows")

    # Refresh count
    dataset = opik_client.get_or_create_dataset(DATASET_NAME)
    print(f"✅ Done. '{DATASET_NAME}' item count:", dataset.dataset_items_count)


name: ai_tutor_coding_practise_dsa_prompts
id: 01a13edd-9d4b-75c2-8522-02eac5c07cf1
item count (before upload): 100
⏭️ Skipping upload: 'ai_tutor_coding_practise_dsa_prompts' already has 100 items. Set FORCE_UPLOAD=True to upload anyway.


## Export Opik → CSV (optional)

Download an existing dataset from Opik to `data/exports/`.


In [ ]:
from datetime import datetime

EXPORT_DATASET_NAME = DATASET_NAME  # change to export a different dataset
RUN_EXPORT = False  # set True to download from Opik

if RUN_EXPORT:
    export_dataset = opik_client.get_or_create_dataset(EXPORT_DATASET_NAME)
    export_df = export_dataset.to_pandas()

    export_dir = REPO_ROOT / "data" / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = export_dir / f"{EXPORT_DATASET_NAME}_{stamp}.csv"
    export_df.to_csv(out_path, index=False)
    print(f"Saved {len(export_df)} rows to:\n  {out_path.resolve()}")
else:
    print("Set RUN_EXPORT=True to export from Opik.")
